# 06 — Target-Aware DifIISR

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shelly-serafimovich/target-aware-infrared-sr/blob/main/notebooks/06_target_aware_difiisr_v2.ipynb)

First implementation pilot of our method. The frozen DifIISR UNet and autoencoder are not retrained. Because official DifIISR inference uses `one_step=True`, we optimize the one-step predicted latent x0 itself through a differentiable image-space target-preservation objective, then decode the refined latent.

**Important:** no GT boxes are used by the guidance. The objective is computed globally from LR-derived candidate locations. GT remains evaluation-only.


## 1. Setup

This notebook is standalone as long as Notebook 05 has already created the saved pilot manifest on Drive. Run on a GPU runtime. It recreates the pinned DifIISR environment and reuses the frozen LR; you do **not** need Notebook 05 open in the same runtime.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os,sys,subprocess,random
import numpy as np,pandas as pd,torch
from PIL import Image

PROJECT_REPO='https://github.com/shelly-serafimovich/target-aware-infrared-sr.git'
DIFIISR_REPO='https://github.com/zirui0625/DifIISR.git'
DIFIISR_COMMIT='09ca97ea48d481656dd8090e84099c963059ac41'
SEED=12345
DATASET_ROOT=Path('/content/drive/MyDrive/thermal-drone-dataset')
LR_DIR=DATASET_ROOT/'difiisr_baseline'/'valid'/'LR_difiisr_train_degradation_v2'
DYN_OUT=DATASET_ROOT/'difiisr_target_dynamics'
TA_OUT=DATASET_ROOT/'target_aware_difiisr'/'pilot_v1'; TA_OUT.mkdir(parents=True,exist_ok=True)
project_dir=Path('/content/target-aware-infrared-sr'); difiisr_dir=Path('/content/DifIISR')

def run(cmd,cwd=None):
    print('$',' '.join(map(str,cmd)))
    subprocess.run(list(map(str,cmd)),cwd=cwd,check=True)

if not project_dir.exists(): run(['git','clone',PROJECT_REPO,project_dir])
if not difiisr_dir.exists(): run(['git','clone',DIFIISR_REPO,difiisr_dir])
run(['git','fetch','--all'],cwd=difiisr_dir)
run(['git','checkout','--detach',DIFIISR_COMMIT],cwd=difiisr_dir)

ENV=Path('/content/difiisr-env'); PY=str(ENV/'bin'/'python')
if not Path(PY).exists():
    run([sys.executable,'-m','pip','install','-q','uv'])
    run([sys.executable,'-m','uv','venv','--python','3.10',ENV])
    run([sys.executable,'-m','uv','pip','install','--python',PY,'torch==2.1.1','torchvision==0.16.1','xformers==0.0.23','numpy==1.25.2','scipy==1.9.3','packaging==24.2','setuptools==69.5.1'])
    run([sys.executable,'-m','uv','pip','install','--python',PY,'pyiqa==0.1.12','basicsr==1.4.2','opencv-python','matplotlib','timm','pandas','pillow','scikit-learn','scikit-image','lpips','loguru','omegaconf','six','tqdm','albumentations','einops','imageio','gdown'])

weights=difiisr_dir/'weights'; weights.mkdir(exist_ok=True)
ckpt=weights/'DifIISR.pth'
if not ckpt.exists(): run([PY,'-m','gdown','1PhRvk1Dlp3CCrPkrxRfNbNZ3fNgviDVZ','-O',ckpt])
ae=weights/'autoencoder_vq_f4.pth'
if not ae.exists():
    run([PY,'-m','gdown','https://github.com/zsyOAOA/ResShift/releases/download/v2.0/autoencoder_vq_f4.pth','-O',ae])

manifest=DYN_OUT/'pilot_targets.csv'
assert manifest.exists(), f'Missing pilot manifest: {manifest}. Run Notebook 05 target selection once.'
print('Frozen LR:',LR_DIR)
print('Pilot manifest:',manifest)
print('Output:',TA_OUT)


## 2. Method v1 — LR-driven candidate mask

We need a zero-shot target-aware signal without GT at inference. For this first pilot we build a soft candidate map directly from the frozen LR image: local positive contrast above a surrounding local mean. The map is upsampled to SR space and treated as a fixed weighting mask.

The refinement objective encourages the decoded predicted x0 to retain **positive local contrast** at LR-supported candidate locations while penalizing excessive global deviation from the original DifIISR prediction. This is deliberately minimal: first establish that a one-step, no-retraining intervention can improve target preservation before adding SCR/Hoyer ablations.


## 3. Create the pilot script


In [ ]:
script=TA_OUT/'run_target_aware_v1.py'
script.write_text(r'''import argparse,sys,random
from pathlib import Path
import numpy as np,pandas as pd,torch
import torch.nn.functional as F
from PIL import Image
from omegaconf import OmegaConf
sys.path.insert(0,'/content/DifIISR')
# Force the VQ autoencoder to use vanilla attention. xFormers inference is fine,
# but its backward kernel is unstable on the Colab T4 stack used for guidance.
import ldm.modules.diffusionmodules.model as ae_model
ae_model.XFORMERS_IS_AVAILBLE=False
from sampler import DifIISRSampler

ap=argparse.ArgumentParser()
ap.add_argument('--manifest'); ap.add_argument('--lr'); ap.add_argument('--out'); ap.add_argument('--config')
ap.add_argument('--seed',type=int,default=12345); ap.add_argument('--steps',type=int,default=4)
ap.add_argument('--lr-guidance',type=float,default=.08); ap.add_argument('--lambda-fid',type=float,default=.15)
a=ap.parse_args()
random.seed(a.seed); np.random.seed(a.seed); torch.manual_seed(a.seed); torch.cuda.manual_seed_all(a.seed)

cfg=OmegaConf.load(a.config)
cfg.model.ckpt_path='/content/DifIISR/weights/DifIISR.pth'; cfg.diffusion.params.sf=4
cfg.autoencoder.ckpt_path='/content/DifIISR/weights/autoencoder_vq_f4.pth'
s=DifIISRSampler(cfg,chop_size=512,chop_stride=448,chop_bs=1,use_fp16=True,seed=a.seed,ddim=True)
df=pd.read_csv(a.manifest); lrdir=Path(a.lr); out=Path(a.out); (out/'SR').mkdir(parents=True,exist_ok=True)

def pad64(y):
 h,w=y.shape[-2:]; ph=(64-h%64)%64; pw=(64-w%64)%64
 return F.pad(y,(0,pw,0,ph),mode='reflect'),h,w

def candidate_mask(y):
 # y is [-1,1], RGB-like thermal. Build mask in LR space without GT.
 g=y.mean(1,keepdim=True)
 mu=F.avg_pool2d(g,7,1,3)
 pos=F.relu(g-mu)
 # robust per-image normalization; suppress weak/background responses.
 flat=pos.flatten(1)
 scale=torch.quantile(flat,.95,dim=1,keepdim=True).view(-1,1,1,1).clamp_min(1e-4)
 m=(pos/scale).clamp(0,1)
 m=F.relu((m-.35)/.65)
 return m

def contrast_objective(img,mask):
 # decoded img in [-1,1]; local contrast at LR-supported candidates.
 g=img.mean(1,keepdim=True)
 mu=F.avg_pool2d(g,31,1,15)
 return (mask*F.relu(g-mu)).sum()/(mask.sum()+1e-6)

rows=[]
for k,r in df.iterrows():
 ip=lrdir/r.image
 arr=np.asarray(Image.open(ip).convert('RGB'),dtype=np.float32)/255.
 y=torch.from_numpy(arr).permute(2,0,1).unsqueeze(0).cuda(); y=(y-.5)/.5
 y,h,w=pad64(y)
 mk_lr=candidate_mask(y)
 mk=F.interpolate(mk_lr,scale_factor=4,mode='bilinear',align_corners=False)

 model_kwargs={'lq':y} if s.configs.model.params.cond_lq else None
 with torch.no_grad():
  z_y=s.base_diffusion.encode_first_stage(y,s.autoencoder,up_sample=True)
  noise=torch.randn_like(z_y); z=s.base_diffusion.prior_sample(z_y,noise)
  i=s.base_diffusion.num_timesteps-1; t=torch.tensor([i],device=y.device)
  o=s.base_diffusion.ddim_sample(model=s.model,x=z,y=z_y,t=t,clip_denoised=False,model_kwargs=model_kwargs)
  z0=o['pred_xstart'].detach()
  base=s.base_diffusion.decode_first_stage(z0,s.autoencoder).detach().clamp(-1,1)

 # Optimize only the predicted latent x0; model weights remain frozen.
 zg=z0.float().detach().requires_grad_(True)
 opt=torch.optim.Adam([zg],lr=a.lr_guidance)
 hist=[]
 for q in range(a.steps):
  opt.zero_grad(set_to_none=True)
  dec=s.base_diffusion.decode_first_stage(zg,s.autoencoder,no_grad=False).float()
  c=contrast_objective(dec,mk)
  fid=F.mse_loss(dec,base.float())
  loss=-c+a.lambda_fid*fid
  loss.backward(); opt.step()
  hist.append((float(c.detach()),float(fid.detach())))
 with torch.no_grad():
  pred=s.base_diffusion.decode_first_stage(zg,s.autoencoder).clamp(-1,1)
 pred=pred[...,:h*4,:w*4]
 im=((pred[0].float().mean(0).cpu().numpy()+1.)*.5*255.).clip(0,255).astype(np.uint8)
 Image.fromarray(im).save(out/'SR'/r.image)
 rows.append(dict(image=r.image,contrast_start=hist[0][0],contrast_end=hist[-1][0],fidelity_end=hist[-1][1]))
 print(k+1,'/',len(df),r.image,hist[-1])
pd.DataFrame(rows).to_csv(out/'pilot_log.csv',index=False)
print('DONE',len(rows))
''')
print(script)


## 4. Run a small validation pilot

Use the same 45-image manifest from Notebook 05. This is only a feasibility run; do not run the full validation yet.


## 4. Stronger guidance pilot — v2

The first pilot changed contrast only negligibly. This run keeps the same frozen model, LR-derived mask and fidelity term, but increases latent refinement strength (8 steps, guidance LR 0.40) on the same 45-image pilot before detector evaluation.


In [ ]:
manifest=DATASET_ROOT/'difiisr_target_dynamics'/'pilot_targets.csv'
cmd=[PY,'-u',str(script),'--manifest',str(manifest),'--lr',str(LR_DIR),'--out',str(TA_OUT),
     '--config','/content/DifIISR/configs/DifIISR_test.yaml','--seed','12345',
     '--steps','8','--lr-guidance','0.40','--lambda-fid','0.15']
env=os.environ.copy(); env['CUDA_VISIBLE_DEVICES']='0'; env['MPLBACKEND']='Agg'
print(' '.join(map(str,cmd)))
result=subprocess.run(list(map(str,cmd)),cwd='/content/DifIISR',env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(result.stdout)
if result.returncode!=0: raise RuntimeError('Target-aware pilot failed; inspect output above.')
log=pd.read_csv(TA_OUT/'pilot_log.csv'); display(log.describe())


## 5. Sanity check before YOLO

First compare a few Original DifIISR and Target-Aware outputs visually and verify that the intervention is local/subtle rather than creating bright artifacts. If this passes, the next notebook step is the frozen-YOLO comparison on the pilot, then validation-scale ablation.


In [ ]:
import matplotlib.pyplot as plt
ORIG=DATASET_ROOT/'difiisr_baseline'/'valid'/'SR_difiisr_train_degradation_v2'
names=pd.read_csv(manifest).image.drop_duplicates().head(5).tolist()
for name in names:
 a=np.asarray(Image.open(ORIG/name).convert('L'))
 b=np.asarray(Image.open(TA_OUT/'SR'/name).convert('L'))
 fig,ax=plt.subplots(figsize=(8,4))
 ax.imshow(np.concatenate([a,b],axis=1),cmap='gray')
 ax.set_title('Original DifIISR  |  Target-Aware v1')
 ax.axis('off'); plt.show()
